In [ ]:
##QDPO
import torch
import torch.nn as nn
import torch.optim as optim
from transformers import AutoTokenizer, AutoModelForCausalLM
from torch.utils.data import DataLoader, Dataset

device = "cuda" if torch.cuda.is_available() else "cpu"

model_name = "microsoft/phi-2"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    output_hidden_states=True,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32
).to(device)

model.train()


class PreferenceDataset(Dataset):
    def __init__(self, data):
        self.data = data

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx]


class HilbertProjection(nn.Module):
    def __init__(self, hidden_size):
        super().__init__()
        self.W = nn.Linear(hidden_size, hidden_size)

    def forward(self, h):
        projected = self.W(h)
        real = torch.cos(projected)
        imag = torch.sin(projected)
        psi = torch.cat([real, imag], dim=-1)
        psi = psi / psi.norm(dim=-1, keepdim=True)
        return psi


def compute_log_prob(model, input_ids, attention_mask):
    outputs = model(input_ids=input_ids, attention_mask=attention_mask)
    logits = outputs.logits[:, :-1, :]
    labels = input_ids[:, 1:]
    log_probs = torch.log_softmax(logits, dim=-1)
    selected = torch.gather(log_probs, 2, labels.unsqueeze(-1)).squeeze(-1)
    return selected.sum(dim=1)


def get_representation(model, input_ids, attention_mask):
    outputs = model(input_ids=input_ids, attention_mask=attention_mask)
    hidden = outputs.hidden_states[-1]
    pooled = hidden.mean(dim=1)
    return pooled


def qdpo_training_loop(preference_data, epochs=3, alpha=0.7, beta=0.1, lr=1e-5):
    dataset = PreferenceDataset(preference_data)
    loader = DataLoader(dataset, batch_size=4, shuffle=True)

    hidden_size = model.config.hidden_size
    projector = HilbertProjection(hidden_size).to(device)

    optimizer = optim.Adam(list(model.parameters()) + list(projector.parameters()), lr=lr)

    for epoch in range(epochs):
        for batch in loader:
            prompts = batch["prompt"]
            chosen = batch["chosen"]
            rejected = batch["rejected"]

            chosen_inputs = tokenizer(prompts, chosen, return_tensors="pt", padding=True, truncation=True).to(device)
            rejected_inputs = tokenizer(prompts, rejected, return_tensors="pt", padding=True, truncation=True).to(device)

            logp_chosen = compute_log_prob(model, chosen_inputs["input_ids"], chosen_inputs["attention_mask"])
            logp_rejected = compute_log_prob(model, rejected_inputs["input_ids"], rejected_inputs["attention_mask"])

            delta_dpo = logp_chosen - logp_rejected

            h_chosen = get_representation(model, chosen_inputs["input_ids"], chosen_inputs["attention_mask"])
            h_rejected = get_representation(model, rejected_inputs["input_ids"], rejected_inputs["attention_mask"])

            psi_chosen = projector(h_chosen)
            psi_rejected = projector(h_rejected)

            overlap = torch.sum(psi_chosen * psi_rejected, dim=-1) ** 2
            delta_q = 1 - overlap

            delta = alpha * delta_dpo + (1 - alpha) * delta_q

            loss = -torch.log(torch.sigmoid(beta * delta)).mean()

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        print(f"Epoch {epoch+1} Loss:", loss.item())

    return model, projector